# 토큰과 중간 진행 스트리밍

사용자가 기다리는 동안 토큰, 노드 완료, 업무 진행률을 어떻게 구분해 보여 줄까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 모델이 포함된 그래프


In [2]:
# 모델 호출 전후에 진행률을 내보내는 한 노드 그래프를 구성합니다.
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    answer: str

model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")

def answer(state: State) -> dict[str, str]:
    writer = get_stream_writer()
    writer({"stage": "model_call", "progress": 30})
    response = model.invoke("두 문장으로 답하세요: " + state["question"])
    writer({"stage": "done", "progress": 100})
    return {"answer": response.text}

builder = StateGraph(State)
builder.add_node("answer", answer)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()
print("그래프 준비 완료")


그래프 준비 완료


## 세 종류의 사건 관찰


In [3]:
# 메시지 사건 수를 세고 글자가 있는 조각만 원래 순서로 합칩니다.
seen = {"messages": 0, "updates": 0, "custom": 0}
preview = []
final_answer = None
for mode, data in graph.stream(
    {"question": "LLM 챗봇이 답변 글자를 생성되는 대로 보여 주면 대기 시간이 짧게 느껴지는 이유는 무엇인가요?", "answer": ""},
    stream_mode=["messages", "updates", "custom"],
):
    seen[mode] += 1
    if mode == "messages":
        chunk, metadata = data
        if chunk.text:
            preview.append(chunk.text)
    else:
        print(mode, data)
        if mode == "updates":
            final_answer = data["answer"]["answer"]
print("messages 사건 수:", seen["messages"])
print("내용이 있는 조각 수:", len(preview))
print("답변 앞부분:", "".join(preview)[:80])
assert seen["messages"] > 0 and seen["updates"] == 1 and seen["custom"] == 2
assert "".join(preview) == final_answer
print("조각을 합친 답변과 최종 상태 일치:", True)


custom {'stage': 'model_call', 'progress': 30}


custom {'stage': 'done', 'progress': 100}
updates {'answer': {'answer': 'LLM이 글자를 생성되는 대로 바로 보여 주면, 사용자는 시스템이 즉시 반응하고 있다는 피드백을 계속 받아 실제 대기 시간이 덜 길게 느껴집니다. 또한 완성된 답을 한꺼번에 기다리는 대신 진행 상황이 보이기 때문에 불확실성과 답답함이 줄어들어 체감 지연이 짧아집니다.'}}
messages 사건 수: 88
내용이 있는 조각 수: 84
답변 앞부분: LLM이 글자를 생성되는 대로 바로 보여 주면, 사용자는 시스템이 즉시 반응하고 있다는 피드백을 계속 받아 실제 대기 시간이 덜 길게 느껴집니다
조각을 합친 답변과 최종 상태 일치: True


## 표시 정책 만들기


In [4]:
# 사건 종류마다 사용자 화면에 표시할 문구를 구분합니다.
labels = {
    "messages": "답변을 작성하고 있습니다",
    "updates": "단계가 완료되었습니다",
    "custom": "업무 진행률이 바뀌었습니다",
}
for mode in ["messages", "updates", "custom"]:
    print(mode, "→", labels[mode])


messages → 답변을 작성하고 있습니다
updates → 단계가 완료되었습니다
custom → 업무 진행률이 바뀌었습니다
